# Module 5: Training with OpenEnv + TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/openenv-course/blob/main/module-5/notebook.ipynb)

⚠️ **Note**: This notebook requires an A100 40GB GPU. It's designed for Google Colab Pro or similar environments.

In this notebook, you'll:
1. Set up the Wordle environment
2. Define reward functions
3. Configure GRPO training
4. Train Qwen3-1.7B to play Wordle
5. Evaluate the trained model

## Hardware Check

In [ ]:
!nvidia-smi

## Setup

Install dependencies. This may take a few minutes.

In [ ]:
!pip install openenv-core trl transformers datasets accelerate torch -q
# For faster training on A100:
# !pip install vllm bitsandbytes -q

## 1. Connect to the Wordle Environment

In [ ]:
import os
import sys

# Clone OpenEnv repo for typed clients
if not os.path.exists('OpenEnv'):
    !git clone https://github.com/meta-pytorch/OpenEnv.git

repo = os.path.abspath('OpenEnv')
sys.path.insert(0, repo)
sys.path.insert(0, os.path.join(repo, 'src'))

In [ ]:
from envs.textarena_env import TextArenaEnv
from envs.textarena_env.models import TextArenaAction

# Connect to hosted Wordle environment
env = TextArenaEnv(base_url="https://burtenshaw-textarena.hf.space")

# Test connection
with env.sync() as sync_env:
    result = sync_env.reset()
    print(f"Environment connected!")
    print(f"Initial prompt: {result.observation.rendered[:200]}...")

## 2. Define Reward Functions

Multiple reward signals help the model learn better strategies.

In [ ]:
def reward_correct(observation) -> float:
    """Reward for solving the puzzle."""
    return 1.0 if observation.get('done') and observation.get('reward', 0) > 0 else 0.0

def reward_greens(observation) -> float:
    """Reward based on number of green (correct position) letters."""
    feedback = observation.get('rendered', '')
    green_count = feedback.count('G')
    return green_count / 5.0  # Normalize by word length

def reward_yellows(observation) -> float:
    """Reward based on number of yellow (wrong position) letters."""
    feedback = observation.get('rendered', '')
    yellow_count = feedback.count('Y')
    return yellow_count / 5.0

def reward_repetition(observation, history) -> float:
    """Penalize repeated guesses."""
    # This would need access to guess history
    # Simplified version for now
    return 1.0

reward_functions = [reward_correct, reward_greens, reward_yellows]

## 3. System Prompt

Guide the model's behavior with a clear system prompt.

In [ ]:
system_prompt = """
You are an expert Wordle solver.

RULES:
- Guess a 5-letter English word
- Feedback: GREEN (correct position), YELLOW (wrong position), GRAY (not in word)
- 6 attempts maximum

RESPONSE FORMAT:
Only respond with your guess in square brackets, e.g., [crane]

STRATEGY:
- Start with vowel-rich words: CRANE, SLATE, STARE
- Use GREEN letters in their positions
- Move YELLOW letters to new positions
- Eliminate GRAY letters
- Never repeat a guess
""".strip()

## 4. GRPO Configuration

Configure the training parameters.

In [ ]:
from trl import GRPOConfig

grpo_config = GRPOConfig(
    output_dir="./wordle-grpo-output",
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=64,
    per_device_train_batch_size=1,
    num_generations=2,
    max_completion_length=8,
    max_prompt_length=1400,
    gradient_checkpointing=True,
    logging_steps=1,
    save_steps=100,
    # For A100 with vLLM:
    # use_vllm=True,
    # vllm_mode="colocate",
    # vllm_gpu_memory_utilization=0.1,
)

print("GRPO Config:")
print(f"  Learning rate: {grpo_config.learning_rate}")
print(f"  Batch size: {grpo_config.per_device_train_batch_size}")
print(f"  Gradient accumulation: {grpo_config.gradient_accumulation_steps}")

## 5. Load Model and Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading model: {model_name}")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded! Parameters: {model.num_parameters():,}")

## 6. Create Training Dataset

A simple dataset with prompts to start Wordle games.

In [ ]:
from datasets import Dataset

# Create a simple dataset
num_training_examples = 100

dataset = Dataset.from_dict({
    "prompt": ["Play Wordle"] * num_training_examples
})

print(f"Dataset created with {len(dataset)} examples")

## 7. Define Rollout Function

The rollout function defines how the model interacts with the environment.

In [ ]:
import re

def extract_guess(text: str) -> str:
    """Extract the guessed word from model output."""
    match = re.search(r'\[(\w{5})\]', text.lower())
    if match:
        return match.group(1)
    return "crane"  # Default guess

def rollout_func(trainer, prompts):
    """Play one Wordle game per prompt."""
    results = []
    
    with env.sync() as sync_env:
        for prompt in prompts:
            # Reset environment
            result = sync_env.reset()
            
            # Play game
            for turn in range(6):  # Max 6 attempts
                if result.observation.done:
                    break
                
                # Build prompt
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": result.observation.rendered}
                ]
                
                # Generate (simplified - in real training, use trainer.generate)
                text = "[crane]"  # Placeholder
                
                # Extract guess and step
                guess = extract_guess(text)
                result = sync_env.step(TextArenaAction(content=f"[{guess}]"))
            
            # Calculate rewards
            obs_dict = result.observation.model_dump()
            rewards = [func(obs_dict) for func in reward_functions]
            
            results.append({
                "reward": sum(rewards) / len(rewards),
                "done": result.observation.done
            })
    
    return results

print("Rollout function defined!")

## 8. Training (Placeholder)

⚠️ **Note**: Full GRPO training requires significant compute and time (~90 minutes on A100).
This is a placeholder showing the structure. For actual training, see the full course materials.

In [ ]:
# Full training code would be:
# from trl import GRPOTrainer
#
# trainer = GRPOTrainer(
#     model=model,
#     args=grpo_config,
#     train_dataset=dataset,
#     tokenizer=tokenizer,
#     reward_funcs=reward_functions,
#     rollout_func=rollout_func,
# )
#
# trainer.train()

print("Training section - see full course materials for complete implementation")

## 9. Evaluate the Model

Test the trained model on Wordle games.

In [ ]:
def evaluate_model(model, tokenizer, num_games=5):
    """Evaluate the model on multiple Wordle games."""
    wins = 0
    total_attempts = 0
    
    with env.sync() as sync_env:
        for game_num in range(num_games):
            print(f"\n{'='*60}")
            print(f"Game {game_num + 1}/{num_games}")
            print('='*60)
            
            result = sync_env.reset()
            attempts = 0
            
            while not result.observation.done and attempts < 6:
                attempts += 1
                
                # In a real implementation, generate with the model
                guess = "crane" if attempts == 1 else "slate"
                
                print(f"\nAttempt {attempts}: [{guess}]")
                result = sync_env.step(TextArenaAction(content=f"[{guess}]"))
                print(result.observation.rendered[-100:])
                
                if result.observation.done:
                    if result.observation.reward and result.observation.reward > 0:
                        wins += 1
                        print(f"✓ Won in {attempts} attempts!")
                    else:
                        print(f"✗ Failed after {attempts} attempts")
                    break
            
            total_attempts += attempts
    
    print(f"\n\n{'='*60}")
    print("RESULTS")
    print('='*60)
    print(f"Win rate: {wins}/{num_games} ({wins/num_games*100:.1f}%)")
    print(f"Average attempts: {total_attempts/num_games:.1f}")
    print('='*60)

# Run evaluation
# evaluate_model(model, tokenizer, num_games=5)

## Key Takeaways

1. **GRPO = Simple RL for LLMs**: No value model needed
2. **Multiple reward signals**: Help the model learn better strategies
3. **Environment as a plugin**: Swap Wordle for any OpenEnv environment
4. **TRL handles complexity**: You focus on environment + rewards

You've completed the OpenEnv course! You now know how to:
- Use existing environments
- Deploy environments
- Build custom environments
- Train LLMs with environment feedback

Go build something awesome!